# Extract transect and make radial topo
## Transect into Seaside OR

In [ ]:
%matplotlib inline

In [ ]:
from pylab import *
from clawpack.geoclaw import topotools, kmltools, dtopotools
from clawpack.visclaw import gridtools, animation_tools
from clawpack.geoclaw.util import haversine, gctransect
from clawpack.clawutil.util import fullpath_import
from scipy.interpolate import interp1d
from scipy.interpolate import RegularGridInterpolator, griddata
import os,sys
NGT = fullpath_import('/Users/rjl/git/AGTwork/geoclaw1d/src_python/nonuniform_grid_tools2.py')

In [ ]:
extent = [-140, -123.85, 45., 48.]
coarsen = 1
arcsec = coarsen * 30
print('Will download etopo22_30s data at %i arcsecond resolution' % arcsec)

In [ ]:
url_thredds = 'https://www.ngdc.noaa.gov/thredds/dodsC/global/ETOPO2022/30s/30s_bed_elev_netcdf/ETOPO_2022_v1_30s_N90W180_bed.nc'
etopo = topotools.read_netcdf(url_thredds, extent=extent,
                             coarsen=coarsen, verbose=True)

In [ ]:
figure(figsize=(12,6))
ax = axes()
etopo.plot(axes=ax, limits=(-3000,1000),
          cb_kwargs={'extend':'both','shrink':0.7})
title('etopo 2022 30" topo');

In [ ]:
j0 = where(topo.y <= 45.993)[0].max()
y0 = topo.y[j0]
print(f'Extracting E-W transect at y = {y0:.5f}')

fig,axs = subplots(2,1,figsize=(10,11))

xtrans = etopo.X[j0,:]  # x along transect
ytrans = etopo.Y[j0,:]  # y along transect (constant)
ztrans = etopo.Z[j0,:]  # topo along transect
ax = axs[0]
ax.plot(etopo.x, ztrans)
ax.grid(True)
#ax.set_ylim(-50,50)
ax.ticklabel_format(useOffset=False)

ax.set_title(f'30" topography on transect at y = {y0:.5f}');


ax = axs[1]
ax.plot(etopo.x, ztrans)

# zoom in on lower plot:

ax = axs[1]
ax.plot(etopo.x, ztrans)
ax.set_ylim(-10,15)
#ax.set_xlim(-124.4, -123.9)
ax.set_xlim(-123.945, -123.91)
ax.set_xlabel('longitude')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Zoom near shore');


## Nearshore topo

In [ ]:
url_thredds = 'https://www.ngdc.noaa.gov/thredds/dodsC/tiles/nthmp/tiled_19as/' \
    + 'ncei19_n46x00_w124x00_2019v1.nc'
extent = [-124,-123.9, 45.97, 46]
coarsen = 3 # subsample from 1/9" to 1/3"
topo13 = topotools.read_netcdf(url_thredds, extent=extent,
                             coarsen=coarsen, verbose=True)

In [ ]:
figure(figsize=(12,6))
ax = axes()
topo13.plot(axes=ax, limits=(-50,50),
          cb_kwargs={'extend':'both','shrink':0.7})
title('CUDEM 1/3" topo');

In [ ]:
y0a =  45.98 #45.991
j0 = where(topo13.y <= y0a)[0].max()
y0 = topo13.y[j0]
print(f'Extracting E-W transect at y = {y0:.5f}')

fig,axs = subplots(2,1,figsize=(10,11))

xtrans = topo13.X[j0,:]  # x along transect
ytrans = topo13.Y[j0,:]  # y along transect (constant)
ztrans = topo13.Z[j0,:]  # topo along transect
ax = axs[0]
ax.plot(topo13.x, ztrans)
ax.grid(True)
ax.set_ylim(-50,50)
ax.ticklabel_format(useOffset=False)

ax.set_title(f'1/3" topography on transect at y = {y0:.5f}');

# zoom in on lower plot:

ax = axs[1]
ax.plot(topo13.x, ztrans)
ax.set_ylim(-10,15)
ax.set_xlim(-123.945, -123.91)
ax.set_xlabel('longitude')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Zoom near shore');

In [ ]:
topos = [etopo, topo13]

In [ ]:
x1trans, x2trans = -136, -123.91
y1trans, y2trans = 46.97992, 45.97992
mxtopo = 200000  # start with finer grid to interpolate from topofiles

xtrans,ytrans = gctransect(x1trans,y1trans, 
                                x2trans,y2trans, mxtopo, 'W')
print(f'Found great circle transect with {len(xtrans)} points')

In [ ]:
ztrans = nan*ones(xtrans.shape)
toposource = -ones(xtrans.shape)

for k,topo in enumerate(topos):
    #topo_fcn =  RegularGridInterpolator((topo.x,topo.y), topo.Z.T,       
    #                     method='linear',bounds_error=False,fill_value=nan)
    #xytrans = vstack((xtrans,ytrans)).T
    topo_fcn = topo.make_function()
    topotrans = topo_fcn(xtrans,ytrans)
    ztrans = where(isnan(topotrans), ztrans, topotrans)
    toposource = where(isnan(topotrans), toposource, k)
    nk = (toposource == k).sum()
    print(f'Extracted {nk} points from topofile {k}')

In [ ]:
if isnan(ztrans).sum() > 0:
    print('*** ztrans has %i nan values' % isnan(ztrans).sum()) 

In [ ]:
figure(figsize=(12,6))
ax = axes()
etopo.plot(axes=ax, limits=(-3000,1000),
           cb_kwargs={'extend':'both','shrink':0.7})
plot(xtrans, ytrans, 'yellow')
title('Great circle transect');

In [ ]:
# convert to meters along transect:
rtrans = haversine(xtrans[0],ytrans[0],xtrans,ytrans)
print(f'Length of transect = {(rtrans[-1]-rtrans[0])/1e3:.3f} km')

# shift away from origin:
#rtrans += 1000e3

topo_fcn_1d = interp1d(rtrans, ztrans, kind='linear', fill_value='extrapolate')

if 0:
    h_min = 20  # switch to uniform fine grid at this depth 
    dx_min = 1. # grid resolution on and near shore
    #dx_max = 500. # maximum in deep ocean, roughly 15"
    rp,zp = NGT.make_celledges_cfl(r[0], r[-1], topo_fcn_1d,
                                   dx_min=dx_min, h_min=h_min, fname='celledges.data',
                                   plot_topo=False)

In [ ]:
xdx_vals = array([[0, 500.], [700e3, 500.], [920e3, 10.], [1100e3, 10.]])
xdx_vals[:,1] /= 5
dx_fcn = interp1d(xdx_vals[:,0], xdx_vals[:,1], kind='linear')
x2 = r[-1]
x = zeros(50000)
for j in range(len(x)):
    x[j+1] = x[j] + dx_fcn(x[j])
    if x[j+1] > x2:
        break
x = x[:(j+1)]
onshore_res = xdx_vals[-1,1]
print(f'Selected {len(x)} points on transect with onshore resolution {onshore_res} m')

redge = x
zedge = topo_fcn_1d(x)


In [ ]:
rlat = 20 + redge / 111e3  # convert to latitudes relative to pole at x=0
rlat_z = vstack((rlat,zedge)).T
fname = f'celledges_{int(onshore_res):d}m.txt'
savetxt(fname, rlat_z, fmt='%.10f', header=f'{len(rlat)}  # number of cell edges', comments='')
print('Created ',fname)

In [ ]:
fig,axs = subplots(2,1,figsize=(10,11))

rkm = redge/1e3

ax = axs[0]
ax.plot(rkm, zedge)
ax.grid(True)
ax.set_title(f'GeoClaw 1D topography on great circle transect with variable spacing');

ax = axs[1]
ax.plot(rkm, zedge)
ax.set_ylim(-10,20)
ax.set_xlim(927, 931)
ax.set_xlabel('radial distance r (km)')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_title('Zoom near shore');

## DTopo file

In [ ]:
dtopofile = '/Users/rjl/B/dtopo/dtopofiles/CSZ_L1-extended-pmel.tt3'
dtopo2d = dtopotools.DTopography(dtopofile, 3)

In [ ]:
dtopo2d_fcn = dtopo2d.make_function()

In [ ]:
dtopo_trans = dtopo2d_fcn(xtrans, ytrans, 10)

In [ ]:
dtopo_fcn_1d = interp1d(rtrans, dtopo_trans, kind='linear', fill_value='extrapolate')

In [ ]:
rdtopo = hstack((0., arange(600e3, 1000e3, 500)))
dz = dtopo_fcn_1d(rdtopo)

In [ ]:
plot(rdtopo/1e3, dz)

In [ ]:
fig,axs = subplots(2,1,figsize=(10,11))

rkm = redge/1e3

ax = axs[0]
ax.plot(rdtopo/1e3, dz)
ax.set_ylim(-10,20)
ax.set_ylabel('dz (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)
ax.set_xlim(700,1000)
ax.set_title('Deformation');

ax = axs[1]
ax.plot(rkm, zedge, 'g')
ax.grid(True)

#ax.set_xlim(927, 931)
ax.set_xlim(700,1000)
ax.set_xlabel('radial distance r (km)')
ax.set_ylabel('elevation (m)')
ax.ticklabel_format(useOffset=False)
ax.grid(True)


In [ ]:
t_rdtopo_dz = vstack((ones(len(rdtopo)), rdtopo, dz)).T
fname = 'L1_dtopo.dtt1'
savetxt(fname, t_rdtopo_dz, fmt='%.10f')

In [ ]:
plot(rlat,zedge)
xlim(

In [ ]:
for factor in [1,2,5,10]:
    xdx_vals = array([[0, 500.], [700e3, 500.], [920e3, 10.], [1100e3, 10.]])
    xdx_vals[:,1] /= factor
    dx_fcn = interp1d(xdx_vals[:,0], xdx_vals[:,1], kind='linear')
    x2 = r[-1]
    x = zeros(50000)
    for j in range(len(x)):
        x[j+1] = x[j] + dx_fcn(x[j])
        if x[j+1] > x2:
            break
    x = x[:(j+1)]
    onshore_res = xdx_vals[-1,1]
    print(f'Selected {len(x)} points on transect with onshore resolution {onshore_res} m')
    
    redge = x
    zedge = topo_fcn_1d(x)

    rlat = 20 + redge / 111e3  # convert to latitudes relative to pole at x=0
    rlat_z = vstack((rlat,zedge)).T
    fname = f'celledges_{int(onshore_res):d}m.txt'
    savetxt(fname, rlat_z, fmt='%.10f', header=f'{len(rlat)}  # number of cell edges', comments='')
    print('Created ',fname)

    # dtopo

    dzedge = dtopo_fcn_1d(redge)
    t_rlat_dz = vstack((ones(len(redge)), rlat, dzedge)).T
    fname = f'L1_dtopo_{int(onshore_res):d}m.dtt1'
    savetxt(fname, t_rlat_dz, fmt='%.10f')
    print('Created ',fname)
